# 导入数据与基本清洗

In [ ]:
import pandas as pd
import numpy  as np
import sqlite3
# 读取
df = pd.read_csv('UserBehavior.csv',names=['user_id','item_id','category_id','behavior','timestamp'],header=None)

# 原始 timestamp 是Unix秒数
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour

# 查看缺失值和重复值
print(df.isnull().sum())
df.drop_duplicates(inplace=True)

# 删除异常时间（如超出范围）
df = df[(df['timestamp'] >= '2017-11-25') & (df['timestamp'] <= '2017-12-03')]

# SQL

In [ ]:
# 清洗后的 DataFrame 导入 SQLite

conn = sqlite3.connect(':memory:')
df.to_sql('user_behavior', conn, index=False, if_exists='replace')

## 用SQL计算PV、UV、跳失率

In [ ]:
query_pv_uv = """
SELECT 
    date,
    COUNT(*) AS pv,
    COUNT(DISTINCT user_id) AS uv,
    ROUND(CAST(COUNT(*) AS FLOAT) / COUNT(DISTINCT user_id), 2) AS avg_pv_per_user
FROM user_behavior
WHERE behavior = 'pv'
GROUP BY date
ORDER BY date;
"""
# 执行SQL并转回dataframe
pv_uv_df = pd.read_sql(query_pv_uv, conn)

## 漏斗分析

In [ ]:
funnel_query = """
SELECT 
    '1-点击' AS stage, COUNT(DISTINCT user_id) AS users FROM user_behavior WHERE behavior='pv'
UNION ALL
SELECT 
    '2-收藏', COUNT(DISTINCT user_id) FROM user_behavior WHERE behavior='fav'
UNION ALL
SELECT 
    '3-加购', COUNT(DISTINCT user_id) FROM user_behavior WHERE behavior='cart'
UNION ALL
SELECT 
    '4-购买', COUNT(DISTINCT user_id) FROM user_behavior WHERE behavior='buy';
"""
funnel_df = pd.read_sql(funnel_query, conn)
funnel_df['转化率'] = funnel_df['users'] / funnel_df['users'].iloc[0] * 100
print(funnel_df)

## 复购率

In [ ]:
repurchase_query = """
WITH user_buy_times AS (
    SELECT user_id, COUNT(*) AS buy_cnt
    FROM user_behavior
    WHERE behavior = 'buy'
    GROUP BY user_id
)
SELECT 
    SUM(CASE WHEN buy_cnt >= 2 THEN 1 ELSE 0 END) AS repurchase_users,
    COUNT(*) AS total_buy_users,
    ROUND(CAST(SUM(CASE WHEN buy_cnt >= 2 THEN 1 ELSE 0 END) AS FLOAT) / COUNT(*), 3) AS repurchase_rate
FROM user_buy_times;
"""
repur_df = pd.read_sql(repurchase_query, conn)

# Python数据分析

## 同期群留存热力图
定义“首日购买”为用户首次购买日期，然后看他们在次日、3日、7日是否再次购买。

In [ ]:
# 先拿到每个用户首次购买日期
first_buy = df[df.behavior=='buy'].groupby('user_id')['date'].min().reset_index()
first_buy.columns = ['user_id','first_date']
df_buy = df[df.behavior=='buy']
df_buy = df_buy.merge(first_buy, on='user_id')
df_buy['day_diff'] = (df_buy['date'] - df_buy['first_date']).dt.days

# 生成留存矩阵
cohort_matrix = df_buy.groupby(['first_date','day_diff']).agg(user_count=('user_id','nunique')).reset_index()
cohort_pivot = cohort_matrix.pivot_table(index='first_date', columns='day_diff', values='user_count')
cohort_pivot = cohort_pivot.div(cohort_pivot[0], axis=0)  # 除以首日人数
# 用 seaborn 画热力图
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10,6))
sns.heatmap(cohort_pivot, annot=True, fmt='.0%', cmap='YlGnBu')
plt.title('用户购买同期群留存热力图')
plt.show()

## RFM 分层
这里 R：最近一次购买距今天数，F：购买频次，M 在数据集中无金额，可用购买次数替代（或假定 M=F）。

In [ ]:
import datetime
snapshot_date = pd.Timestamp('2017-12-04')
buy_data = df[df.behavior=='buy']
rfm = buy_data.groupby('user_id').agg(
    last_buy=('date', 'max'),
    freq=('user_id', 'count')
).reset_index()
rfm['recency'] = (snapshot_date - pd.to_datetime(rfm['last_buy'])).dt.days
rfm['F_score'] = pd.qcut(rfm['freq'], 3, labels=[1,2,3])
rfm['R_score'] = pd.qcut(rfm['recency'].rank(method='first'), 3, labels=[3,2,1])  # 注意越近分越高
rfm['RFM'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str)
print(rfm.groupby('RFM')['user_id'].count())